In [0]:
#Test Data# 25
'''api_stream_1 = iter([
    {"source": "api", "record_id": "A001", "customer_id": 101, "amount": 25.50},
    {"source": "api", "record_id": "A002", "customer_id": 102, "amount": 40.00},
])

file_stream_1 = iter([
    {"source": "file", "record_id": "F001", "customer_id": 103, "amount": 15.75},
    {"source": "file", "record_id": "F002", "customer_id": 104, "amount": 99.99},
])

db_partition_1 = iter([
    {"source": "database", "record_id": "D001", "customer_id": 105, "amount": 120.00},
    {"source": "database", "record_id": "D002", "customer_id": 106, "amount": 0.00},
])

empty_api_stream = iter([])

edge_case_stream = iter([
    {"source": "api", "record_id": "A004", "customer_id": 0, "amount": 0},
    {"source": "file", "record_id": "F005", "customer_id": 999999999999, "amount": 999999999.99},
    {"source": "database", "record_id": "D004", "customer_id": 112, "amount": None},
    {"source": "api", "record_id": "A005", "customer_id": 113, "amount": 0.01},
    {"source": "file", "record_id": "F006", "customer_id": 114, "amount": 10.123456},
])

invalid_record_stream = iter([
    {"source": "api", "record_id": None, "customer_id": 108, "amount": 50.00},
    {"source": "file", "customer_id": 109, "amount": 25.00},  # record_id missing
    {"source": "database", "record_id": "D003", "customer_id": "abc", "amount": 75.00},
    {"source": "api", "record_id": "A003", "customer_id": 110, "amount": -15.00},
    {"source": "file", "record_id": "F004", "customer_id": 111, "amount": "not-a-number"},
])

file_stream_with_data = iter([
    {"source": "file", "record_id": "F003", "customer_id": 107, "amount": 10.00},
])

empty_db_partition = iter([])

duplicate_stream_1 = iter([
    {"source": "api", "record_id": "A006", "customer_id": 115, "amount": 30.00},
])

duplicate_stream_2 = iter([
    {"source": "file", "record_id": "A006", "customer_id": 115, "amount": 30.00},
]) '''

In [0]:
#Design a Memory-Efficient Multi-Source Iterator with itertools.chain-Build the processing logic for thousands of lazily generated record streams originating from APIs, files, and database partitions. Process them as one logical stream without loading all records into memory, while isolating failures from individual sources.

#import
import re
from collections.abc import Iterator, Mapping, Iterable
from itertools import chain
from typing import TypeVar

MAX_RECORD_LENGTH=1024
RECORD_KEYS:tuple[str, ...]=("source", "record_id", "customer_id", "amount")
SOURCES:tuple[str,...]=("api", "file", "database")
RECORD_ID_PATTERN=re.compile(r"[A-Z]\d{3}")
RECORD_ID_SET:set[str]=set()
T=TypeVar("T")

'''api_stream_1 = iter([
    {"source": "api", "record_id": "A001", "customer_id": 101, "amount": 25.50},
    {"source": "api", "record_id": "A002", "customer_id": 102, "amount": 40.00},
])'''
#Validate Record
def record_validation(record:Mapping[str,Iterable["T"]], stream_index:int)->dict:
    #check if record is dictionary/Iterable
    if not isinstance(record, dict):
        raise TypeError(
            f"record expected to be key-value pair received {type(record).__name__}")    

    #check if all the keys are present in record
    for key in RECORD_KEYS:
        #listing dictionary keys
        key_list=record.get(key,())

        #check length of key_list
        if len(key_list)==0:
            raise ValueError(
                f"record is empty for the stream: {stream_index}")
        if len(key_list)!=len(RECORD_KEYS):
            raise ValueError(
                f"expected to have {len(RECORD_KEYS)} keys received {len(key_list)} for the stream: {stream_index}")  
            
        #unpacking dictionary values
        source, record_id, customer_id, amount=record.values()
        #check value of key Source
        if source not in SOURCES:
            raise ValueError(f"Unknown source {source!r} for the stream: {stream_index}")

        #check value of key-record_id
        if not RECORD_ID_PATTERN.fullmatch(record_id):
            raise ValueError(f"record_id not valid for the stream: {stream_index}")
        # check if record is duplicate
        if record_id in RECORD_ID_SET:
            raise ValueError(f"record is {record_id} already exists")
        RECORD_ID_SET.add(record_id) #adding it to set to track it in future

        #check value of key-customer_id
        if not isinstance(customer_id,int):
            raise TypeError(f"customer id expected to be number received {type(customer_id).__name__}for the stream: {stream_index}")
        #check value of key-amount is float
        if not isinstance(amount, float):
            raise TypeError(f"amount expected to be float value received {type(amount).__name__} for the stream: {stream_index}")
        #check value of key-amount is non negative & valid
        if amount<0:
            raise ValueError(f"amount cannot be negative for the stream: {stream_index}")   
        
    return record

#Validate Stream/partition
def stream_validation(stream:object, stream_index:int)->Iterator[str]:
    #check if record is iterable/Not none
    if not isinstance(stream, Iterable):
        raise TypeError(f"Stream expected to be iterable received {type(stream).__name__} for strem index {stream_index}")

    #check if record is not string/bytesarray/bytes
    if isinstance(stream,(str, bytes, bytearray, Mapping)):
        raise TypeError(f"stream expected to be list received {type(stream).__name__} for strem index {stream_index}")     
    
    #yield valid stream
    for record in stream:
        yield record_validation(record, stream_index)

#Process Stream/partition
def stream_processing(stream_batch:Iterable[object] )->Iterator[str]:
    #iterate over stream
    validate_stream=(
        stream_validation(stream, stream_index)
        for stream_index, stream in enumerate(stream_batch)  )
    return chain.from_iterable(validate_stream)

#Main function 
if __name__=="__main__":
    api_stream_1 = iter([
    {"source": "api", "record_id": "A001", "customer_id": 101, "amount": 25.50},
    {"source": "api", "record_id": "A002", "customer_id": 102, "amount": 40.00},
])
for key in stream_processing(api_stream_1):
    print(key)

In [0]:
#Design a Memory-Efficient Multi-Source Iterator with itertools.chain-Build the processing logic for thousands of lazily generated record streams originating from APIs, files, and database partitions. Process them as one logical stream without loading all records into memory, while isolating failures from individual sources.
#import
import re
from itertools import chain
from collections.abc import Mapping, Iterator, Iterable
from typing import Any
from decimal import Decimal, InvalidOperation
import logging
logger=logging.getLogger(__name__)

#global variables
VALID_SOURCES=frozenset({"api", "file","database"})
REQUIRED_KEYS=frozenset({"source", "record_id", "customer_id", "amount"})
REORD_ID_PATTERN=re.compile(r"^[A-Z]\d{3}$")

Record=dict[str, Any]
# sample record-{"source": "database", "record_id": "D001", "customer_id": 105, "amount": 120.00}
#record validation & normalization
def record_validation(record:object, stream_index:int)->Record:
    #check if record is mapping or not
    if not isinstance(record, Mapping):
        raise TypeError(f"record of stream index {stream_index} expected to be key-value pair", f"received {type(record).__name__}")
    #check if record is empty

    #check if record is duplicate &  #check if all keys present 
    actual_keys=set(record.keys())
    missing_keys=REQUIRED_KEYS-actual_keys
    unexpected_keys=actual_keys-REQUIRED_KEYS    
    
    if missing_keys or unexpected_keys:
        raise ValueError(f"record of stream index {stream_index} have invalid keys",
                         f"Missing={sorted(missing_keys)}  unexpected={sorted(unexpected_keys)}")

    #unpacking
    source=record["source"]
    record_id=record["record_id"]
    customer_id=record["customer_id"]
    amount=record["amount"]

    #check source is string & present in predefined set
    if not isinstance(source, str) or source not in VALID_SOURCES:
        raise ValueError(
            f"record of stream index {stream_index} has invalid source name")
    #check record id is string and follows pattern
    if not isinstance(record_id, str) or not REORD_ID_PATTERN.fullmatch(record_id) :
        raise TypeError(
            f"record of stream index {stream_index} has invalid record_id{record_id!r}")
    #check customer id is non boolean and integer
    if isinstance(customer_id, bool) or not isinstance(customer_id, int):
        raise TypeError(
            f"record of stream index {stream_index} has invalid customer id {customer_id!r}")
    if customer_id<0:
        raise ValueError(f"customer id cannot be negative")
    #check if amount is decimal, 
    if isinstance(amount, bool):
        raise TypeError("amount cannot be boolean value")
    try:
        normalized_amount=Decimal(str(amount))
    except(InvalidOperation, ValueError) as exc:
        raise ValueError(
            f"record of stream index {stream_index} has non numeric amount {amount!r}")
    #amount is non negative & finite number
    if not normalized_amount.is_finite() or normalized_amount<0:
        raise ValueError(f"record of stream index {stream_index} has invalid amount value")
    #return normalized records
    return {
        "source":source,
        "record_id": record_id,
        "customer_id":customer_id,
        "amount":amount
    }
#stream validation
def stream_validation(stream:object, stream_index:int)->Iterator[Record]:
    if isinstance(stream,(bytearray,bytes, str, Mapping)) or not isinstance(stream, Iterable):
        raise(f"stream with index {stream_index} expected to be iterable received{type(stream).__name__}")
    try:
        for record in stream:
            yield record_validation(record,stream_index)
    except Exception as exc:
        logger.exception(
            "skipping reamining records as stream failed",
            stream_index, exc
        )    
#process stream
def process_stream(streams : Iterable[object])->Iterator[Record]:
    valid_streams=(
        stream_validation(stream, stream_index)
        for stream_index, stream in enumerate(streams)
    )
    yield from chain.from_iterable(valid_streams)
#main function
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)

    api_stream_1 = iter([
        {"source": "api", "record_id": "A001", "customer_id": 101, "amount": 25.50},
        {"source": "api", "record_id": "A002", "customer_id": 102, "amount": 40.00},
    ])

    file_stream_1 = iter([
        {"source": "file", "record_id": "F001", "customer_id": 103, "amount": "-1"},
    ])

    all_streams = [api_stream_1, file_stream_1]

    for record in process_stream(all_streams):
        print(record) 

In [0]:
'''# Positive test case: same number of columns and expected types
source_columns_positive = [
    "customer_id",
    "customer_name",
    "signup_date",
    "is_active",
    "account_balance",
] 
expected_data_types_positive = [
    "integer",
    "string",
    "date",
    "boolean",
    "decimal",
] 

# Negative test case: fewer data types than source columns
source_columns_negative_missing_type = [
    "customer_id",
    "customer_name",
    "signup_date",
] 
expected_data_types_negative_missing_type = [
    "integer",
    "string",
] 

# Negative test case: unsupported / invalid expected type
source_columns_negative_invalid_type = [
    "customer_id",
    "customer_name",
    "signup_date",
] 
expected_data_types_negative_invalid_type = [
    "integer",
    "varchar2_typo",
    "date",
] 

# Negative test case: duplicate column names
source_columns_negative_duplicate_column = [
    "customer_id",
    "customer_name",
    "customer_id",
] 
expected_data_types_negative_duplicate_column = [
    "integer",
    "string",
    "integer",
]

# Negative test case: empty column name
source_columns_negative_empty_column = [
    "customer_id",
    "",
    "signup_date",
]
expected_data_types_negative_empty_column = [
    "integer",
    "string",
    "date",
]

# Edge test case: both lists are empty
source_columns_edge_empty = []
expected_data_types_edge_empty = []

# Edge test case: one nullable-looking field name
source_columns_edge_nullable_field = [
    "customer_id",
    "middle_name",
    "deleted_at",
]
expected_data_types_edge_nullable_field = [
    "integer",
    "string",
    "timestamp",
]

# Edge test case: unusual but valid column names
source_columns_edge_special_names = [
    "_ingested_at",
    "customer_id_v2",
    "address_line_1",
    "is_vip_customer",
]
expected_data_types_edge_special_names = [
    "timestamp",
    "integer",
    "string",
    "boolean",
]

# Edge test case: case sensitivity
source_columns_edge_case_sensitive = [
    "Customer_ID",
    "customer_id",
]
expected_data_types_edge_case_sensitive = [
    "integer",
    "string",
]

# Edge test case: large schema input
source_columns_edge_large = [f"attribute_{i}" for i in range(1, 1001)]
expected_data_types_edge_large = ["string"] * 1000
'''

In [0]:
#Combine Column Names and Data Types with zip-Given one list of source column names and another list of expected data types, create column-to-type mappings for use by a schema validation process. Detect and handle situations where the two lists have different lengths.
#import
import re
from itertools import zip
from Iterables import Iterator, Iterator, Mapping

#global variables
COLUMN_NAME_STANDARD=re.compile(r"([_]*[a-z]+_[a-z]+)+")
VALID_DATA_TYPES=["integer", "string", "timestamp", "boolean","date", "decimal"]
MAX_SCHEMA_LENGTH=100

#column validation & normalization
def column_validation(column_name:object, col_index:int)->str:
    #column name is str
    if not isinstance(column_name, str):
        raise TypeError(f"column name expected to be string received {type(column_name).__name__} ",
                        f"column index is {col_index}")
    #column name is empty
    normalized_column=column_name.strip()
    normalized_column=normalized_column.lower()
    if not normalized_column:
        raise ValueError(f"column name cannot be empty",
                         f"column index is {col_index}")
    #number of column not exceeding max length
    if col_index>MAX_SCHEMA_LENGTH:
        raise ValueError(f"number of columns exceeded to {MAX_SCHEMA_LENGTH}",
                         f"column index is {col_index}")
    #column name matches with pattern
    if not COLUMN_NAME_STANDARD.fullmatch(normalized_column):
        raise ValueError(f"column name is not up to standard",
                         f"column index is {col_index}")
    return (normalized_column)

#Data Type Validation
def data_type_validation(data_type:object, dtype_index:int)->str:
    #data type name is string 
    if not isinstance(data_type, str):
        raise TypeError(f"data type name expected to be string received {type(data_type).__name__}",
                        f"data type name index is {dtype_index}")
    #data type name is not empty
    normalized_dtype_name=data_type.strip()
    normalized_dtype_name=normalized_dtype_name.lower()
    if not normalized_dtype_name:
        raise ValueError(f"data type name cannot be empty",
                         f"data type name index is {dtype_index}")
    #data type name not exceeding maximum length
    if dtype_index>MAX_SCHEMA_LENGTH:
        raise ValueError(f"number of data types exceeded maximum allowed data types",
                         f"data type name index is {dtype_index}")
    #data type name is part of predefined list
    if normalized_dtype_name not in VALID_DATA_TYPES:
        raise ValueError(f"Data types is invalid",
                         f"data type name index is {dtype_index}"
                         f"data type name is {normalized_dtype_name}")
    return (normalized_dtype_name,dtype_index)
#schema validation & Zip column names with data types
def schema_validation(schema_col:object,schema_dtype:object)->tuple:
    #check if schema is list
    if not isinstance(schema_col, list):
        raise TypeError(f"colmn schema is expected to be list",
                        f"received {type(schema_col).__name__!r}")
        
    if not isinstance(schema_dtype, list):
        raise TypeError(f"data type schema is expected to be list",
                        f"received {type(schema_dtype).__name__!r}")
    #check if schema is empty
    if len(schema_col)==0 or len(schema_dtype)==0:
        raise ValueError(f"Schema is invalid")
    #check if number of columns=number of data_types
    if len(schema_col)!= len(schema_dtype):
        raise ValueError(f"Number of columns not matches with number of data types",
                         f"number of columns={len(schema_col)}",
                         f"number of data types={len(schema_dtype)}") 

    #zip column_names with data_types
    return zip((column_validation(schema_col)),data_type_validation(schema_dtype))    

#main function
if __name__=="__main__":
    # Positive test case: same number of columns and expected types
source_columns = [
    "customer_id",
    "customer_name",
    "signup_date",
    "is_active",
    "account_balance",
]
expected_data_types = [
    "integer",
    "string",
    "date",
    "boolean",
    "decimal",
]
result=schema_validation(source_columns,expected_data_types)
print(result)

In [0]:
#from __future__ import annotations
#Combine Column Names and Data Types with zip-Given one list of source column names and another list of expected data types, create column-to-type mappings for use by a schema validation process. Detect and handle situations where the two lists have different lengths.
#import

import re
from typing import Final
from collections.abc import Sequence
#global variables
VALID_DATATYPES:Final[frozenset[str]]=frozenset({"integer", "string", "timestamp", "boolean", "date", "decimal"})
MAX_SCHEMA_LENGTH:Final=100
COLUMN_NAME_PATTERN:Final[re.Pattern[str]]=re.compile(r"^[a-z][a-z0-9_]*$")
#column name validation
def normaize_column_name(column_name:object, index:int)->str:
    if not isinstance(column_name, str):
        raise TypeError(
            f"column at index {index} must be string"
            f"received {type(column_name).__name__}")
    normalized_name=column_name.lower().strip()
    if not normalized_name:
        raise ValueError(
            f"column at index {index} cannot be empty")
    if not COLUMN_NAME_PATTERN.fullmatch(normalized_name):
        raise ValueError(
            f"Invalid column name at index {index}:{column_name!r}"
            "use lowercase letters, digits, & underscores"
            "name must start with letter")
    return normalized_name 
#data type validaion
def normalize_datatype_name(dtype:object, index:int)->str:
    if not isinstance(dtype, str):
        raise TypeError(
            f"data type at index {index} must be string"
            f"received {type(dtype).__name__}")
        
    normalized_type=dtype.lower().strip()
    if normalized_type not in VALID_DATATYPES:
        raise ValueError(
            f"unsupported data types at index {index}: {type(dtype).__name__}"
            f"Allowed vlaues: {VALID_DATATYPES}")
    return normalized_type 
#schema mapping
def build_schema_mapping(
    source_columns:Sequence[object],
    expected_data_types:Sequence[object]
)->dict[str, str]:
    if (not isinstance(source_columns, Sequence) or isinstance(source_columns, (str, bytes))):
        raise TypeError(
            "source column must be list or tuple")
    if (not isinstance(expected_data_types, Sequence) or isinstance(expected_data_types, (str, bytes))):
        raise TypeError(
            "expected_data_types must be list or tuple")
    if not source_columns or not expected_data_types:
        raise ValueError(f"source columns or data types cannot be empty")
    if len(source_columns) !=len(expected_data_types):
        raise ValueError(
            "column and data type must mantch"
            f"received {len(source_columns)} columns and"
            f"{len(expected_data_types)} data types")
    if len(source_columns)>MAX_SCHEMA_LENGTH:
        raise ValueError(
            f"schema has {len(source_columns)} columns and"
            f"maximum allowed length is {MAX_SCHEMA_LENGTH}")
    schema_mapping:dict[str, str]={}
    for index,(column_name, data_type) in enumerate(
        zip(source_columns, expected_data_types, strict=True)
    ):
        normalized_name=normaize_column_name(column_name, index)
        normalized_datatype=normalize_datatype_name(data_type, index)

        if normalized_name in schema_mapping:
            raise ValueError(
                f"duplicate column name detected at index {index}"
                f"{normalized_name!r}")
        schema_mapping[normalized_name]=normalized_datatype
    return schema_mapping

#main function 
if __name__=="__main__":
    source_columns = [
        "customer_id",
        "customer_name",
        "signup_date",
        "is_active",
        "account_balance",
    ]

    expected_data_types = [
        "integer",
        "string",
        "date",
        "boolean",
        "decimal",
    ]
    schema=build_schema_mapping(source_columns, expected_data_types)
    print(schema)

In [0]:
'''Test case	source_tables	source_counts	target_counts	Expected reconciliation records
Positive: all counts match	["customers", "orders"]	[100, 250]	[100, 250]	customers: PASS, orders: PASS
Negative: one mismatch	["customers", "orders"]	[100, 250]	[100, 245]	customers: PASS, orders: FAIL
Negative: all mismatch	["customers", "orders"]	[100, 250]	[99, 249]	customers: FAIL, orders: FAIL
Edge: empty inputs	[]	[]	[]	[]
Edge: source table has zero rows	["daily_clicks"]	[0]	[0]	daily_clicks: PASS
Negative: source zero, target has rows	["daily_clicks"]	[0]	[5]	daily_clicks: FAIL
Negative: target zero, source has rows	["daily_clicks"]	[5]	[0]	daily_clicks: FAIL
Edge: source count missing for last table	["customers", "orders", "payments"]	[100, 250]	[100, 250, 80]	customers: PASS, orders: PASS, payments: FAIL / missing source count
Edge: target count missing for last table	["customers", "orders", "payments"]	[100, 250, 80]	[100, 250]	customers: PASS, orders: PASS, payments: FAIL / missing target count
Edge: both counts missing for last table	["customers", "orders", "payments"]	[100, 250]	[100, 250]	customers: PASS, orders: PASS, payments: FAIL / missing source and target count
Edge: extra source count with no table	["customers"]	[100, 999]	[100]	customers: PASS; extra count should be ignored or flagged, depending on program rules
Edge: extra target count with no table	["customers"]	[100]	[100, 999]	customers: PASS; extra count should be ignored or flagged, depending on program rules
Edge: table name missing/blank	["customers", "", "orders"]	[100, 20, 250]	[100, 20, 250]	customers: PASS, blank-name record should be FAIL / invalid table name, orders: PASS
Edge: duplicate table name	["customers", "orders", "customers"]	[100, 250, 100]	[100, 250, 100]	Three ordered records; all PASS unless duplicates are explicitly invalid
Negative: negative source count	["refunds"]	[-5]	[-5]	refunds: FAIL / invalid negative count
Negative: negative target count	["refunds"]	[5]	[-5]	refunds: FAIL / invalid negative count
Negative: non-integer source count	["orders"]	["250"]	[250]	orders: FAIL / invalid source count type
Negative: non-integer target count	["orders"]	[250]	[250.5]	orders: FAIL / invalid target count type
Edge: None count	["inventory"]	[None]	[0]	inventory: FAIL / missing source count
Edge: large counts	["event_logs"]	[9_223_372_036_854_775_807]	[9_223_372_036_854_775_807]	event_logs: PASS
Negative: large-count mismatch	["event_logs"]	[9_223_372_036_854_775_807]	[9_223_372_036_854_775_806]	event_logs: FAIL
'''

In [0]:
#Compare Source and Target Row Counts with zip-Given ordered lists of source table names, source row counts, and target row counts, generate reconciliation records indicating whether each table passed or failed count validation. Handle missing entries in any input collection.
#import
from collections.abc import Sequence, Iterable
#from iterators import Any
from typing import Final, Any
#global variable
TABLE_NAME_PATTERN:Final[re.Pattern[str]]=re.compile(r"[a-zA-Z][a-z0-9_]*$")
#table & count validation
def table_validation(table_name:Any, index:int)->str:
     #check if source table name is string
    if not isinstance(table_name, str):
        raise TypeError(
            f"table name of index {index} expected to be string"
            f"received {type(table_name).__name__}")
     #check if table name is empty
    normalize_table_name=table_name.strip().lower()
    if not normalize_table_name:
        raise ValueError(
            f"table name of index {index} cannot be empty")
     #check if table name matches with patterns
    if not TABLE_NAME_PATTERN.fullmatch(normalize_table_name):
        raise ValueError(
            f"table name of index {index} is invalid")
    #return table name
    return normalize_table_name
#count validation
def source_count_validation(source_count:Any, index:int)->int:
    #check if source count is integer
    #check if number is empty
    if source_count is None :
        raise TypeError(
            f"record count of index {index} expected to be integer"
            f"received None")       
    if not isinstance(source_count, int):
        raise TypeError(
            f"record count of index {index} expected to be integer"
            f"received {type(source_count).__name__}")    
    #check if number is non negative or infinite
    if source_count<0 :
        raise ValueError(
            f"record count of index {index} cannot be negative"
            f"received {source_count}" )
    #return source count
    return source_count

def target_count_validation(target_count:Any, index:int)->int:
    if target_count is None :
        raise TypeError(
            f"record count of index {index} expected to be integer"
            f"received None")  
    #check if target count is integer
    if not isinstance(target_count, int):
        raise TypeError(
            f"record count of index {index} expected to be integer"
            f"received {type(target_count).__name__}") 
    #check if number is non negative
    if target_count<0 :
        raise ValueError(
            f"record count of index {index} cannot be negative"
            f"received {target_count}" )
    #return target count
    return target_count
#zip them & compare results
def compare_results(
    source_tables:Sequence[object],
    source_counts:Sequence[object],
    target_counts:Sequence[object])->dict[str, str]:
    #check if any of these are non sequence
    if (not isinstance (source_tables, Sequence) ) or \
       (not isinstance (source_counts, Sequence))  or \
       (not isinstance (target_counts, Sequence))  or \
        isinstance(source_tables,(str,bytes,bytearray)):
           raise TypeError(
               f"one of the sequence is non iterable")
    result_dict:dict[str,str]={}
    message="Pass"
    '''for index,(column_name, data_type) in enumerate(
        zip(source_columns, expected_data_types, strict=True)
    ):'''
    for index, (src_tbl, msg) in enumerate(
        zip(source_tables, message, strict=True)
    ):
        #compare source count & target count sequnce are equal or not
        if source_counts!=target_counts:
            result_dict[source_tables]=["Failed"]            
        #compare length of source table, source count & target count sequnce
        if (len(source_tables)>len(source_counts)):
            result_dict[source_tables]=["source tables are more than total record counts"] 
        if (len(source_tables)<len(source_counts)):
            result_dict[source_tables]=["source tables are less than total record counts"]
        if (len(target_counts)<len(source_counts)):
            result_dict[source_tables]=["target counts are less than total source record counts"]
        if (len(target_counts)>len(source_counts)):
            result_dict[source_tables]=["target counts are more than total source record count" ]       
    #check if source table sequence has duplicate table names
        if source_tables in result_dict:
            result_dict[source_tables]=["This is duplicate table"]    
    result_dict[source_tables]=["Passed"]
    return result_dict

#main function
if __name__=="__main__":
    source_tables=["customers", "orders"]
    source_counts=[100, 250]
    target_counts=[100, 250]

    count_validation=compare_results(source_tables,source_counts,target_counts)
    print(count_validation) 

In [0]:
#Compare Source and Target Row Counts with zip-Given ordered lists of source table names, source row counts, and target row counts, generate reconciliation records indicating whether each table passed or failed count validation. Handle missing entries in any input collection.